# 📈 AI Stock Market Analyst - Trading Advisor Agent

## 🎯 Project Overview

**Goal:** Build an intelligent trading advisor that provides actionable insights for stock market decisions.

**Key Features:**
- 📊 Conditional RAG: Automatically decides when to use documents vs LLM knowledge
- 🎯 Relevance Scoring: Smart decision-making based on similarity scores
- 📈 Quantitative Evaluation: Compare RAG vs LLM performance with metrics
- 💼 Real Trading Advice: Answer questions about market events and trends

---

## 🛠️ Required Packages

### 📦 Core Dependencies

- **langchain**: LangChain framework (chain composition)
- **langchain-community**: PDF loaders and integrations
- **langchain-openai**: OpenAI API integration
- **docarray**: Pure Python vector search (stable, no dependency conflicts)
- **pypdf**: PDF text extraction
- **docx2txt**: DOCX text extraction
- **tiktoken**: Token counting (for cost estimation)
- **openai**: Official OpenAI SDK
- **python-dotenv**: Environment variable management

✅ Install once to run locally or in Colab  
⚠️ Reinstall if session resets

In [ ]:
# Cell 1
# %pip install --upgrade pip
# %pip install -U langchain langchain-community langchain-openai
# %pip install -U docarray pypdf docx2txt tiktoken openai python-dotenv
# %pip install -U sentence-transformers  # 🔑 For FinBERT embeddings

## 🔐 OpenAI API Key Setup
This code automatically loads the OpenAI API Key from the `.env` file.

In [ ]:
# Cell 3
import os
from dotenv import load_dotenv

load_dotenv()
if os.environ.get("OPENAI_API_KEY"):
    print("✅ OpenAI API key loaded successfully from .env file")
else:
    print("❌ Warning: OPENAI_API_KEY not found in .env file")

## ⚙️ Configuration Settings (Hyperparameters)

Below are the main hyperparameters to customize the Trading Advisor system.

**🔬 Hyperparameters:** Configuration values that control system behavior (not learned from data)

---

### 📊 Hyperparameters Worth Testing

**🔴 HIGH IMPACT (Test these first!)**

1. **RELEVANCE_THRESHOLD (0.65)** - Most important!
   - What it does: Decides when to use RAG vs Smart Fallback
   - Test range: 0.50 - 0.80
   - Impact: Higher = stricter (more LLM), Lower = more lenient (more RAG)

2. **FALLBACK_THRESHOLD (0.50)** - Second most important!
   - What it does: Decides when to use LLM vs Smart Fallback
   - Test range: 0.40 - 0.60
   - Impact: Higher = larger fallback zone, Lower = direct LLM more often

3. **LLM_CONFIDENCE_THRESHOLD (5)** - Controls "No Answer" responses
   - What it does: Minimum confidence to give an answer
   - Test range: 3 - 7
   - Impact: Higher = stricter (more "No Answer"), Lower = more permissive

**🟡 MEDIUM IMPACT**

4. **CHUNK_SIZE (800)** - Affects retrieval quality
   - What it does: Size of document chunks for embedding
   - Test range: 400 - 1200
   - Impact: Smaller = precise, Larger = more context

5. **TOP_K_DOCUMENTS (4)** - Number of documents to retrieve
   - Test range: 2 - 8
   - Impact: More docs = more context but also more noise

**🟢 LOW IMPACT (Fine-tuning)**

6. **LLM_TEMPERATURE (0.3)** - Response creativity
   - Test range: 0.0 - 0.7
   - Impact: Higher = more creative/varied, Lower = more focused/consistent

7. **CHUNK_OVERLAP (100)** - Context preservation
   - Test range: 50 - 200
   - Impact: Higher = better context but more redundancy

---

### 🧪 Suggested Experiment Plan

**Phase 1: Threshold Testing (Most Important!)**
- Test RELEVANCE_THRESHOLD: [0.55, 0.60, 0.65, 0.70, 0.75]
- Test FALLBACK_THRESHOLD: [0.40, 0.45, 0.50, 0.55]
- Observe how many questions go to RAG vs LLM vs Fallback

**Phase 2: Confidence Testing**
- Test LLM_CONFIDENCE_THRESHOLD: [3, 5, 7]
- See when system says "No Answer"

**Phase 3: Retrieval Quality**
- Test CHUNK_SIZE: [400, 800, 1200]
- Test TOP_K_DOCUMENTS: [2, 4, 6]

**Phase 4: Fine-tuning**
- Test LLM_TEMPERATURE: [0.0, 0.3, 0.5]
- Test CHUNK_OVERLAP: [50, 100, 200]

---

### 🔍 What to Observe When Testing

**For each hyperparameter change, check:**

1. **Mode Distribution** (Cell 15 output)

2. **Score Patterns**
   - Relevance Scores for each question
   - RAG Score vs LLM Score (in fallback mode)
   - Overall improvement percentage (Cell 19)

3. **Answer Quality** (Cell 21)
   - Are RAG answers better than before?
   - Are LLM answers being used appropriately?
   - Do answers make sense?

4. **Evaluation Metrics** (Cell 19)
   - Overall Score improvement
   - Specificity, Relevance, Factuality scores

---

### 💡 Pro Tips for Experimentation

1. **Change ONE hyperparameter at a time** - easier to see impact
2. **Run all 3 test questions** each time - consistent comparison
3. **Document your findings** - keep notes on what works best
4. **Expected behavior:**
   - Lower RELEVANCE_THRESHOLD → More RAG answers
   - Lower FALLBACK_THRESHOLD → Larger "uncertain zone" (more comparisons)
   - Higher FALLBACK_THRESHOLD → Smaller "uncertain zone" (more direct LLM)
   - Higher LLM_CONFIDENCE → More "No Answer" responses

---

### 📋 Quick Reference Table

| Hyperparameter | Default | Impact | Test Range | What Happens When You INCREASE It |
|---------------|---------|--------|------------|-----------------------------------|
| **RELEVANCE_THRESHOLD** | 0.65 | 🔴 High | 0.50-0.80 | Stricter RAG → More questions go to Fallback/LLM |
| **FALLBACK_THRESHOLD** | 0.50 | 🔴 High | 0.40-0.60 | Larger fallback zone → More questions compared |
| **LLM_CONFIDENCE_THRESHOLD** | 5 | 🔴 High | 3-7 | More "No Answer" responses (stricter) |
| **CHUNK_SIZE** | 800 | 🟡 Medium | 400-1200 | More context per chunk, fewer total chunks |
| **TOP_K_DOCUMENTS** | 4 | 🟡 Medium | 2-8 | More documents retrieved (more context/noise) |
| **LLM_TEMPERATURE** | 0.3 | 🟢 Low | 0.0-0.7 | More creative/varied responses |
| **CHUNK_OVERLAP** | 100 | 🟢 Low | 50-200 | Better context preservation, more redundancy |

**🎯 Start your experiments with the 🔴 High Impact parameters!**

In [ ]:
# Cell 5
# ========================================
# 📌 USER CONFIGURATION (Hyperparameters)
# ========================================

# 1️⃣ Document Settings
DOCS_FOLDER = "../docs"  # Folder containing market analysis files (PDFs and DOCX)

# 2️⃣ Document Splitting Settings (🟡 Medium Impact)
CHUNK_SIZE = 800        # Size of each chunk (characters) | Test: 400-1200
CHUNK_OVERLAP = 100     # Overlap between chunks (characters) | Test: 50-200

# 3️⃣ LLM Model Settings
LLM_MODEL = "gpt-4o-mini"  
LLM_TEMPERATURE = 0.2      # 🟢 Low Impact | Lower = focused, Higher = creative | Test: 0.0-0.7

# 4️⃣ Conditional RAG Settings (Smart Fallback) - 🔴 HIGH IMPACT!
RELEVANCE_THRESHOLD = 0.65  # 🔴 MOST IMPORTANT! | RAG-only threshold | Test: 0.50-0.80
FALLBACK_THRESHOLD = 0.50   # 🔴 2nd MOST IMPORTANT! | Smart Fallback zone | Test: 0.40-0.60
OFF_TOPIC_THRESHOLD = 0.15  # 🔴 3rd IMPORTANT! | Off-topic rejection threshold | Test: 0.15-0.30
                            # Score >= 0.65: Use RAG only (high confidence)
                            # Score 0.50-0.65: Try BOTH, compare, use better answer
                            # Score 0.20-0.50: LLM with domain check (stock-related or reject)
                            # Score < 0.20: Immediate rejection (clearly off-topic)
TOP_K_DOCUMENTS = 4         # 🟡 Medium Impact | Number of docs to retrieve | Test: 2-8

# 5️⃣ LLM Confidence Settings (for NO_ANSWER detection) - 🔴 HIGH IMPACT!
LLM_CONFIDENCE_THRESHOLD = 5  # 🔴 Controls "No Answer" | Test: 3-7
                              # If LLM confidence < 5, return "Information not available"
                              # Lower = stricter (more "no answer"), Higher = more permissive

# 5.5️⃣ Sigmoid Transformation Settings (for Relevance Score Amplification) - 🔴 HIGH IMPACT!
SIGMOID_MIDPOINT = 0.5       # 🔴 CRITICAL! | Center point of sigmoid | Test: 0.50-0.60
                              # Scores above this → boosted high (0.80-0.99)
                              # Scores below this → pushed low (0.01-0.20)
SIGMOID_STEEPNESS = 10        # 🔴 CRITICAL! | How sharp the transition is | Test: 10-25
                              # Higher = sharper separation between relevant/irrelevant
                              # Lower = smoother transition

# 6️⃣ Test Questions (for evaluation)
TEST_QUERIES = [
    # 1. 🟢 TIER 1: RAG Mode (relevance >= RELEVANCE_THRESHOLD = 0.65)
    # Should use documents directly (specific facts from docs)
    "What were the main economic events and Federal Reserve decisions during Trump's second term?",
    
    # 2. 🟢/🔵 TIER 2: Smart Fallback (FALLBACK_THRESHOLD ≤ relevance < RELEVANCE_THRESHOLD, i.e., 0.50-0.65)
    # Should compare RAG vs LLM (broader analysis)
    # Expected: Green if RAG wins, Blue if LLM wins
    "List the major companies mentioned in the October 2025 market review with their performance",
    
    # 3. 🟠 TIER 3: LLM Domain Check (OFF_TOPIC_THRESHOLD ≤ relevance < FALLBACK_THRESHOLD, i.e., 0.20-0.50)
    # Should use LLM with domain check (general stock knowledge, not in docs)
    # Expected: LLM answers (Orange color) - Stock-related
    "How should I trade in highly volatile stock market?",
    
    # 4. 🟠/🔴 TIER 3: LLM Domain Check (OFF_TOPIC_THRESHOLD ≤ relevance < FALLBACK_THRESHOLD, i.e., 0.20-0.50)
    # Should use LLM with domain check, but may have low confidence (prediction is uncertain)
    # Expected: Orange if confidence ≥ 6/10 (answers with caveats), Red if confidence < 6/10 (NO_ANSWER mode)
    "What will AAPL stock price be in 2026?",
    
    # 5. 🔴 TIER 3: LLM Domain Check (OFF_TOPIC_THRESHOLD ≤ relevance < FALLBACK_THRESHOLD, i.e., 0.20-0.50)
    # Should use LLM with domain check, LLM detects off-topic and rejects it
    # Expected: Red color - Domain-based rejection (not stock-related)
    "Who will be the Apple CEO in 2030?",
    
    # 6. 🔴 TIER 4: Off-Topic Auto-Reject (relevance < OFF_TOPIC_THRESHOLD = 0.20)
    # Should immediately reject without LLM call (clearly not about stock trading)
    # Expected: Red color - Auto-reject
    "Explain the concept of quantum entanglement and its applications in quantum computing."
]

# ========================================
# 🛡️ VALIDATION CHECKS
# ========================================

# Check: FALLBACK_THRESHOLD must be < RELEVANCE_THRESHOLD
if FALLBACK_THRESHOLD >= RELEVANCE_THRESHOLD:
    raise ValueError(
        f"\n❌ ERROR: Invalid threshold configuration!\n"
        f"   FALLBACK_THRESHOLD ({FALLBACK_THRESHOLD}) must be < RELEVANCE_THRESHOLD ({RELEVANCE_THRESHOLD})\n"
        f"\n"
        f"   Why? The 'uncertain zone' is defined as:\n"
        f"   [FALLBACK_THRESHOLD, RELEVANCE_THRESHOLD)\n"
        f"\n"
        f"   Current settings would create an impossible zone:\n"
        f"   [{FALLBACK_THRESHOLD}, {RELEVANCE_THRESHOLD}) = INVALID\n"
        f"\n"
        f"   💡 Suggested fix:\n"
        f"   - Keep RELEVANCE_THRESHOLD = {RELEVANCE_THRESHOLD}\n"
        f"   - Set FALLBACK_THRESHOLD to something like {RELEVANCE_THRESHOLD - 0.10:.2f} or {RELEVANCE_THRESHOLD - 0.15:.2f}\n"
    )

# Validate OFF_TOPIC_THRESHOLD
if OFF_TOPIC_THRESHOLD >= FALLBACK_THRESHOLD:
    raise ValueError(
        f"\n❌ ERROR: OFF_TOPIC_THRESHOLD ({OFF_TOPIC_THRESHOLD}) must be < FALLBACK_THRESHOLD ({FALLBACK_THRESHOLD})\n"
        f"   Thresholds must be: OFF_TOPIC < FALLBACK < RELEVANCE\n"
    )

# Validate sigmoid parameters
if not (0.0 <= SIGMOID_MIDPOINT <= 1.0):
    raise ValueError(f"❌ SIGMOID_MIDPOINT must be between 0.0 and 1.0, got {SIGMOID_MIDPOINT}")
if SIGMOID_STEEPNESS <= 0:
    raise ValueError(f"❌ SIGMOID_STEEPNESS must be positive, got {SIGMOID_STEEPNESS}")

print("✅ Configuration Complete!")
print("✅ Validation passed: OFF_TOPIC < FALLBACK < RELEVANCE")
print(f"🎯 Thresholds: OFF_TOPIC={OFF_TOPIC_THRESHOLD}, FALLBACK={FALLBACK_THRESHOLD}, RELEVANCE={RELEVANCE_THRESHOLD}")
print(f"✅ Sigmoid parameters: Midpoint={SIGMOID_MIDPOINT}, Steepness={SIGMOID_STEEPNESS}")
print(f"📂 Documents Folder: {DOCS_FOLDER}")
print(f"📏 Chunk Size: {CHUNK_SIZE} (Overlap: {CHUNK_OVERLAP})")
print(f"🤖 LLM Model: {LLM_MODEL} (Temp: {LLM_TEMPERATURE})")
print(f"🎯 RAG Threshold: {RELEVANCE_THRESHOLD} 🔴")
print(f"🔄 Fallback Threshold: {FALLBACK_THRESHOLD} 🔴")
print(f"🧠 LLM Confidence Threshold: {LLM_CONFIDENCE_THRESHOLD} 🔴")
print(f"📊 Top K Documents: {TOP_K_DOCUMENTS}")
print("\n💡 Tip: Change 🔴 HIGH IMPACT parameters first to see biggest differences!")

## 📊 Logging, Monitoring & Error Handling Setup

**Why add this before hyperparameter tuning?**

When experimenting with different hyperparameters, we need to:
1. 📝 **Log** what happens (for debugging)
2. ⏱️ **Monitor** performance (response time, cost)
3. 🛡️ **Handle errors** gracefully (API failures, timeouts)

---

### 🎯 What We'll Track

**Performance Metrics:**
- ⏱️ Response time per query
- 💰 Token usage and estimated cost
- 📊 Mode distribution (RAG vs LLM vs Fallback)

**Error Handling:**
- ⚠️ OpenAI API errors (rate limits, timeouts)
- 📁 Document loading errors (missing files, corrupt PDFs)
- 🔄 Automatic retry for transient failures

**Experiment Tracking:**
- 💾 Auto-save results to CSV file
- 📈 Compare different hyperparameter configurations
- 🎯 Find optimal settings based on data

---

### 📋 Implementation Level

**✅ What we WILL add (Mid-level):**
- Basic try-except error handling
- Clear error messages
- Simple retry logic (1-2 attempts)
- Performance logging
- CSV export for experiments

In [ ]:
# Cell 6
import logging
import time
import csv
import os
from datetime import datetime
from functools import wraps
import tiktoken

# ========================================
# 📝 LOGGING SETUP
# ========================================

# Create logs directory if it doesn't exist
os.makedirs("logs", exist_ok=True)

# Configure logging
# This will log to both file and console
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(f'logs/trading_advisor_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'),
        logging.StreamHandler()  # Also print to console
    ]
)

logger = logging.getLogger(__name__)

# ========================================
# 💰 COST TRACKING
# ========================================

class CostTracker:
    """
    Track token usage and estimated API costs.
    
    OpenAI pricing (as of 2024):
    - GPT-4o-mini: $0.150 per 1M input tokens, $0.600 per 1M output tokens
    """
    
    # Pricing per 1M tokens (USD)
    PRICING = {
        "gpt-5": {"input": 1.25, "output": 10.00},
        "gpt-5-mini": {"input": 0.25, "output": 2.00},
        "gpt-4o-mini": {"input": 0.15, "output": 0.60},        
    }
    
    def __init__(self, model_name="gpt-4o-mini"):
        """Initialize cost tracker for a specific model."""
        self.model_name = model_name
        self.encoding = tiktoken.encoding_for_model(model_name)
        self.total_input_tokens = 0
        self.total_output_tokens = 0
    
    def count_tokens(self, text):
        """Count tokens in a text string."""
        return len(self.encoding.encode(text))
    
    def add_tokens(self, input_text, output_text):
        """
        Add token counts for input and output.
        
        Args:
            input_text: Input prompt text
            output_text: Model response text
        """
        input_tokens = self.count_tokens(input_text)
        output_tokens = self.count_tokens(output_text)
        
        self.total_input_tokens += input_tokens
        self.total_output_tokens += output_tokens
        
        return input_tokens, output_tokens
    
    def get_cost(self):
        """Calculate total cost in USD."""
        if self.model_name not in self.PRICING:
            return 0.0  # Unknown model
        
        pricing = self.PRICING[self.model_name]
        input_cost = (self.total_input_tokens / 1_000_000) * pricing["input"]
        output_cost = (self.total_output_tokens / 1_000_000) * pricing["output"]
        
        return input_cost + output_cost
    
    def get_summary(self):
        """Get summary of token usage and cost."""
        return {
            "input_tokens": self.total_input_tokens,
            "output_tokens": self.total_output_tokens,
            "total_tokens": self.total_input_tokens + self.total_output_tokens,
            "estimated_cost_usd": self.get_cost()
        }
    
    def reset(self):
        """Reset token counters."""
        self.total_input_tokens = 0
        self.total_output_tokens = 0

# Initialize global cost tracker
cost_tracker = CostTracker(model_name=LLM_MODEL)

# ========================================
# 🛡️ ERROR HANDLING UTILITIES
# ========================================

def retry_on_api_error(max_retries=2, delay=2):
    """
    Decorator to retry function on API errors.
    
    Args:
        max_retries: Maximum number of retry attempts
        delay: Delay in seconds between retries
    
    Usage:
        @retry_on_api_error(max_retries=2, delay=2)
        def api_call():
            # Your API call here
            pass
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            
            for attempt in range(max_retries + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exception = e
                    error_type = type(e).__name__
                    
                    # Log the error
                    if attempt < max_retries:
                        logger.warning(f"⚠️ {error_type} on attempt {attempt + 1}/{max_retries + 1}: {str(e)}")
                        logger.info(f"🔄 Retrying in {delay} seconds...")
                        time.sleep(delay)
                    else:
                        logger.error(f"❌ Failed after {max_retries + 1} attempts: {str(e)}")
            
            # If all retries failed, raise the last exception
            raise last_exception
        
        return wrapper
    return decorator

def safe_file_load(file_path, loader_class):
    """
    Safely load a file with error handling.
    
    Args:
        file_path: Path to file
        loader_class: LangChain loader class (PyPDFLoader or Docx2txtLoader)
    
    Returns:
        List of loaded documents, or empty list if failed
    """
    try:
        loader = loader_class(file_path)
        documents = loader.load()
        logger.info(f"✅ Loaded: {os.path.basename(file_path)} ({len(documents)} pages/sections)")
        return documents
    except FileNotFoundError:
        logger.error(f"❌ File not found: {file_path}")
        return []
    except PermissionError:
        logger.error(f"❌ Permission denied: {file_path}")
        return []
    except Exception as e:
        logger.error(f"❌ Error loading {os.path.basename(file_path)}: {type(e).__name__} - {str(e)}")
        return []

# ========================================
# 📊 EXPERIMENT TRACKING
# ========================================

class ExperimentTracker:
    """Track and save hyperparameter experiment results."""
    
    def __init__(self, csv_filename="hyperparameter_experiments.csv"):
        """Initialize experiment tracker."""
        self.csv_filename = csv_filename
        self.current_experiment = {}
        
        # Create CSV file with headers if it doesn't exist
        if not os.path.exists(csv_filename):
            with open(csv_filename, 'w', newline='') as f:
                writer = csv.DictWriter(f,                 fieldnames=[
                    'timestamp', 'relevance_threshold', 'fallback_threshold', 'off_topic_threshold',
                    'llm_confidence_threshold', 'sigmoid_midpoint', 'sigmoid_steepness',
                    'chunk_size', 'top_k', 'temperature', 'avg_response_time_sec', 'total_tokens',
                    'estimated_cost_usd', 'mode_rag_count', 'mode_fallback_count',
                    'mode_llm_domain_check_count', 'mode_off_topic_count',
                    'mode_no_answer_count', 'mode_error_count', 
                    'avg_improvement_pct', 'improvement_std', 'improvement_min', 'improvement_max',
                    'avg_relevance_score', 'avg_rag_score', 'avg_llm_score',
                    'avg_rag_specificity', 'avg_rag_relevance', 'avg_rag_factuality',
                    'avg_llm_specificity', 'avg_llm_relevance', 'avg_llm_factuality',
                    'notes'
                ])
                writer.writeheader()
    
    def start_experiment(self, config):
        """
        Start tracking a new experiment.
        
        Args:
            config: Dictionary with hyperparameter settings
        """
        self.current_experiment = {
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            **config,
            'response_times': [],
            'mode_counts': {
                'RAG': 0, 
                'FALLBACK': 0, 
                'LLM_DOMAIN_CHECK': 0, 
                'OFF_TOPIC': 0,
                'NO_ANSWER': 0,
                'ERROR': 0
            }
        }
        logger.info(f"🧪 Starting experiment with config: {config}")
    
    def log_query(self, response_time, mode):
        """Log a single query result."""
        self.current_experiment['response_times'].append(response_time)
        self.current_experiment['mode_counts'][mode] += 1
    
    def save_experiment(self, improvement_pct=None, 
                       improvement_std=None, improvement_min=None, improvement_max=None,
                       avg_relevance_score=None, avg_rag_score=None, avg_llm_score=None,
                       avg_rag_specificity=None, avg_rag_relevance=None, avg_rag_factuality=None,
                       avg_llm_specificity=None, avg_llm_relevance=None, avg_llm_factuality=None,
                       notes=""):
        """
        Save experiment results to CSV with detailed metrics.
        
        Args:
            improvement_pct: Overall RAG improvement percentage (from evaluation)
            improvement_std: Standard deviation of improvement across queries
            improvement_min: Minimum improvement across queries
            improvement_max: Maximum improvement across queries
            avg_relevance_score: Average relevance score (0-1)
            avg_rag_score: Average RAG answer quality (1-10)
            avg_llm_score: Average LLM answer quality (1-10)
            avg_rag_specificity: Average RAG specificity score (1-10)
            avg_rag_relevance: Average RAG relevance score (1-10)
            avg_rag_factuality: Average RAG factuality score (1-10)
            avg_llm_specificity: Average LLM specificity score (1-10)
            avg_llm_relevance: Average LLM relevance score (1-10)
            avg_llm_factuality: Average LLM factuality score (1-10)
            notes: Any additional notes about this experiment
        """
        cost_summary = cost_tracker.get_summary()
        
        # Calculate averages
        avg_response_time = sum(self.current_experiment['response_times']) / len(self.current_experiment['response_times']) if self.current_experiment['response_times'] else 0
        
        # Prepare row data
        row = {
            'timestamp': self.current_experiment['timestamp'],
            'relevance_threshold': self.current_experiment.get('relevance_threshold'),
            'fallback_threshold': self.current_experiment.get('fallback_threshold'),
            'off_topic_threshold': self.current_experiment.get('off_topic_threshold'),
            'llm_confidence_threshold': self.current_experiment.get('llm_confidence_threshold'),
            'sigmoid_midpoint': self.current_experiment.get('sigmoid_midpoint'),
            'sigmoid_steepness': self.current_experiment.get('sigmoid_steepness'),
            'chunk_size': self.current_experiment.get('chunk_size'),
            'top_k': self.current_experiment.get('top_k'),
            'temperature': self.current_experiment.get('temperature'),
            'avg_response_time_sec': f"{avg_response_time:.2f}",
            'total_tokens': cost_summary['total_tokens'],
            'estimated_cost_usd': f"${cost_summary['estimated_cost_usd']:.4f}",
            'mode_rag_count': self.current_experiment['mode_counts']['RAG'],
            'mode_fallback_count': self.current_experiment['mode_counts']['FALLBACK'],
            'mode_llm_domain_check_count': self.current_experiment['mode_counts']['LLM_DOMAIN_CHECK'],
            'mode_off_topic_count': self.current_experiment['mode_counts']['OFF_TOPIC'],
            'mode_no_answer_count': self.current_experiment['mode_counts']['NO_ANSWER'],
            'mode_error_count': self.current_experiment['mode_counts']['ERROR'],
            'avg_improvement_pct': f"{improvement_pct:.1f}%" if improvement_pct else "N/A",
            'improvement_std': f"{improvement_std:.2f}%" if improvement_std is not None else "N/A",
            'improvement_min': f"{improvement_min:.1f}%" if improvement_min is not None else "N/A",
            'improvement_max': f"{improvement_max:.1f}%" if improvement_max is not None else "N/A",
            'avg_relevance_score': f"{avg_relevance_score:.3f}" if avg_relevance_score is not None else "N/A",
            'avg_rag_score': f"{avg_rag_score:.2f}" if avg_rag_score is not None else "N/A",
            'avg_llm_score': f"{avg_llm_score:.2f}" if avg_llm_score is not None else "N/A",
            'avg_rag_specificity': f"{avg_rag_specificity:.2f}" if avg_rag_specificity is not None else "N/A",
            'avg_rag_relevance': f"{avg_rag_relevance:.2f}" if avg_rag_relevance is not None else "N/A",
            'avg_rag_factuality': f"{avg_rag_factuality:.2f}" if avg_rag_factuality is not None else "N/A",
            'avg_llm_specificity': f"{avg_llm_specificity:.2f}" if avg_llm_specificity is not None else "N/A",
            'avg_llm_relevance': f"{avg_llm_relevance:.2f}" if avg_llm_relevance is not None else "N/A",
            'avg_llm_factuality': f"{avg_llm_factuality:.2f}" if avg_llm_factuality is not None else "N/A",
            'notes': notes
        }
        
        # Append to CSV
        with open(self.csv_filename, 'a', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=row.keys())
            writer.writerow(row)
        
        logger.info(f"💾 Experiment saved to {self.csv_filename}")
        logger.info(f"📊 Summary: Avg time={avg_response_time:.2f}s, Cost=${cost_summary['estimated_cost_usd']:.4f}, Improvement={improvement_pct:.1f}%")

# Initialize global experiment tracker
experiment_tracker = ExperimentTracker()

print("✅ Logging, Monitoring & Error Handling configured successfully!")
print(f"📝 Logs will be saved to: logs/")
print(f"📊 Experiment results will be saved to: hyperparameter_experiments.csv")
print(f"💰 Cost tracking enabled for model: {LLM_MODEL}")

## 📖 How to Use Logging, Monitoring & Error Handling

### 🎯 What Was Added?

**✅ Automatic Logging:**
- All operations are logged to `logs/trading_advisor_[timestamp].log`
- Logs include timestamps, error messages, and performance metrics

**✅ Cost Tracking:**
- Token usage is automatically counted for all queries
- Estimated API costs are calculated in real-time
- Summary displayed after each experiment

**✅ Error Handling:**
- API failures automatically retry (up to 2 times)
- File loading errors are caught and logged
- User-friendly error messages displayed

**✅ Experiment Tracking:**
- All hyperparameter experiments saved to `hyperparameter_experiments.csv`
- Includes: response time, token usage, cost, mode distribution, improvement %
- Easy to compare different settings in Excel

---

### 💡 How to Use It

**1. Normal Usage (Automatic):**
```python
# Just run the notebook as usual!
# Monitoring happens automatically in the background
```

**2. Check Logs:**
```python
# Logs are saved in: logs/trading_advisor_[timestamp].log
# Open the file to see detailed operation logs
```

**3. View Experiment Results:**
```python
# After running Cell 15 and Cell 19, check:
# hyperparameter_experiments.csv
# Open in Excel to compare different hyperparameter settings
```

**4. Monitor Real-time Cost:**
```python
# Check token usage and cost at any time:
cost_summary = cost_tracker.get_summary()
print(f"Total tokens: {cost_summary['total_tokens']}")
print(f"Estimated cost: ${cost_summary['estimated_cost_usd']:.4f}")
```

---

### 🔍 Understanding the CSV Output

The `hyperparameter_experiments.csv` file contains:

| Column | Description |
|--------|-------------|
| `timestamp` | When the experiment was run |
| `relevance_threshold` | RAG threshold setting (🔴 HIGH IMPACT) |
| `fallback_threshold` | Smart Fallback threshold (🔴 HIGH IMPACT) |
| `llm_confidence_threshold` | Confidence threshold for "No Answer" |
| `chunk_size` | Document chunk size |
| `top_k` | Number of documents retrieved |
| `temperature` | LLM temperature |
| `avg_response_time_sec` | Average response time per query |
| `total_tokens` | Total tokens used (input + output) |
| `estimated_cost_usd` | Estimated API cost in USD |
| `mode_rag_count` | How many queries used RAG mode |
| `mode_fallback_count` | How many used Smart Fallback |
| `mode_llm_count` | How many used LLM-only |
| `avg_improvement_pct` | RAG improvement over LLM-only (%) |
| `improvement_std` | Standard deviation of improvement (consistency) |
| `improvement_min` | Minimum improvement across queries |
| `improvement_max` | Maximum improvement across queries |
| `avg_relevance_score` | Average relevance score (0-1) |
| `avg_rag_score` | Average RAG answer quality (1-10) |
| `avg_llm_score` | Average LLM answer quality (1-10) |
| `avg_rag_specificity` | Average RAG specificity (1-10) |
| `avg_rag_relevance` | Average RAG relevance (1-10) |
| `avg_rag_factuality` | Average RAG factuality (1-10) |
| `avg_llm_specificity` | Average LLM specificity (1-10) |
| `avg_llm_relevance` | Average LLM relevance (1-10) |
| `avg_llm_factuality` | Average LLM factuality (1-10) |

**💡 Use this CSV to find optimal hyperparameters!**

---

### 📈 Understanding Stability Metrics

**Why Std/Min/Max matter:**

When choosing hyperparameters, don't just look at `avg_improvement_pct`!

**Example:**
```
Setting A: avg = +80%, std = 5%, min = +75%, max = +85%
Setting B: avg = +85%, std = 30%, min = +20%, max = +150%

Which is better?
→ Setting A! (Lower std = more consistent)
```

**Interpretation:**
- **Low Std (< 10%)**: 🟢 Very stable, consistent across all queries
- **Medium Std (10-20%)**: 🟡 Acceptable, some variation
- **High Std (> 20%)**: 🔴 Unstable, unpredictable performance

**Pro tip:** In Excel, create a scatter plot:
- X-axis: `avg_improvement_pct`
- Y-axis: `improvement_std`
- Look for points in the **top-left** (high avg, low std) ← **OPTIMAL!**

---

### 🎓 What You've Learned

By adding these features, you now have:
- ✅ **Production-ready logging** (Mid-level skill)
- ✅ **Performance monitoring** (Essential for optimization)
- ✅ **Basic error handling** (Deployable code quality)
- ✅ **Experiment tracking** (Scientific approach to tuning)

## 📁 Document Loading (Local Environment)

This notebook runs in your local environment (not Google Colab).

**How it works:**
- **No file upload needed**: Market analysis files are already in the `docs/` folder
- **Multiple formats**: The system will automatically load all PDF and DOCX files from the folder
- **Automatic processing**: All documents will be combined into a single knowledge base

**Documents loaded:**
- 📊 Weekly market summaries from multiple sources (Claude, ChatGPT, Gemini)
- 📈 Covering October 2025 market analysis and outlook
- 📄 Supports both PDF and DOCX formats

In [ ]:
# Cell 7
# Google Colab file upload is not needed for local environments
# The 'google.colab' library is only available in Google Colab
# from google.colab import files
# uploaded = files.upload()

print("✅ Running in local environment - skipping file upload step")

## 📄 Document Loading and Chunking

Load all documents (PDF and DOCX) from the `docs/` folder and split them into chunks suitable for LLM processing.

**Process:**
1. **Find all files**: Scan the docs folder for PDF and DOCX files
2. **Load documents**: Convert documents to text using appropriate loaders
   - PDFs: PyPDFLoader
   - DOCX: Docx2txtLoader
3. **Split into chunks**: Divide text into manageable pieces (size = CHUNK_SIZE)
4. **Add overlap**: Include overlap between chunks to maintain context (CHUNK_OVERLAP)

**Why chunking?**
- LLMs have token limits
- Smaller chunks improve retrieval accuracy
- Overlap prevents loss of context at boundaries

In [ ]:
# Cell 9

import os
import glob
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Find all PDF and DOCX files in the docs folder
logger.info(f"📂 Scanning folder: {DOCS_FOLDER}")

# Check if docs folder exists
if not os.path.exists(DOCS_FOLDER):
    logger.error(f"❌ Docs folder not found: {DOCS_FOLDER}")
    raise FileNotFoundError(f"Documents folder '{DOCS_FOLDER}' does not exist")

pdf_files = glob.glob(os.path.join(DOCS_FOLDER, "*.pdf"))
docx_files = glob.glob(os.path.join(DOCS_FOLDER, "*.docx"))

print(f"📂 Found {len(pdf_files)} PDF files and {len(docx_files)} DOCX files in {DOCS_FOLDER}/")
logger.info(f"Found {len(pdf_files)} PDFs and {len(docx_files)} DOCX files")

# Check if any files found
if len(pdf_files) == 0 and len(docx_files) == 0:
    logger.warning(f"⚠️ No PDF or DOCX files found in {DOCS_FOLDER}/")
    print(f"⚠️ WARNING: No documents found. Please add PDF or DOCX files to {DOCS_FOLDER}/")

# Load all documents (PDF + DOCX) using safe_file_load
all_documents = []

# Load PDF files with error handling
print(f"\n📄 Loading PDF files...")
for pdf_file in pdf_files:
    documents = safe_file_load(pdf_file, PyPDFLoader)
    all_documents.extend(documents)

# Load DOCX files with error handling
print(f"\n📄 Loading DOCX files...")
for docx_file in docx_files:
    documents = safe_file_load(docx_file, Docx2txtLoader)
    all_documents.extend(documents)

# Check if any documents were successfully loaded
if len(all_documents) == 0:
    logger.error("❌ No documents were successfully loaded")
    raise ValueError("Failed to load any documents. Please check file formats and permissions.")

print(f"\n✅ Total documents loaded: {len(all_documents)}")
logger.info(f"Successfully loaded {len(all_documents)} document sections")

# Split documents into chunks using configuration settings
try:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, 
        chunk_overlap=CHUNK_OVERLAP
    )
    docs = text_splitter.split_documents(all_documents)
    print(f"🔹 Total chunks created: {len(docs)}")
    logger.info(f"Created {len(docs)} text chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")
except Exception as e:
    logger.error(f"❌ Error splitting documents: {str(e)}")
    raise

## 🧠 Embedding and Vector Store Creation

Convert document chunks into vector embeddings using OpenAI's embedding model, then store them in a vector database.

**Process:**
1. **OpenAIEmbeddings()**: Converts each text chunk into a high-dimensional vector (1536 dimensions)
2. **DocArrayInMemorySearch**: Pure Python vector search library
   - Stable, no native library dependencies
   - Memory-based for fast operation
   - Ideal for prototyping and learning
3. **Similarity Search**: Find relevant documents by comparing vector similarity

**Why embeddings?**
- Captures semantic meaning of text
- Enables similarity-based retrieval
- Goes beyond keyword matching

In [ ]:
# Cell 11
import os
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import DocArrayInMemorySearch

# 🔑 Initialize FinBERT embeddings (finance-specific BERT model)
# Why FinBERT instead of OpenAI embeddings?
# - FinBERT is trained on financial texts (Bloomberg, Reuters, etc.)
# - Better at distinguishing finance vs non-finance queries
# - More accurate semantic understanding for stock market domain
try:
    print("🔄 Loading FinBERT model... (first time may take 1-2 minutes)")
    
    # Detect best device for your machine
    # MPS: Apple Silicon (M1/M2/M3 Macs) - fast GPU acceleration
    # CUDA: NVIDIA GPUs - very fast
    # CPU: Universal fallback - slower but works everywhere
    import torch
    if torch.backends.mps.is_available():
        device = 'mps'  # Apple Silicon GPU
        print("✅ Using Apple MPS (Metal Performance Shaders) for acceleration")
    elif torch.cuda.is_available():
        device = 'cuda'  # NVIDIA GPU
        print("✅ Using CUDA (NVIDIA GPU) for acceleration")
    else:
        device = 'cpu'  # CPU fallback
        print("ℹ️ Using CPU (slower, but works on all machines)")
    
        # 🏦 Using FINANCE-SPECIFIC sentence embeddings
    # baconnier/Finance2_embedding_small_en-V1.5 is trained on financial datasets
    # This provides better separation between finance and non-finance queries than general embeddings
    # Next step: Add sigmoid transformation to amplify the relevance scores
    embeddings = HuggingFaceEmbeddings(
        model_name="baconnier/Finance2_embedding_small_en-V1.5",  # Finance-specific embeddings
        model_kwargs={'device': device},  # Auto-detect best device
        encode_kwargs={'normalize_embeddings': True}  # Normalize for better similarity scores
    )
    
    print(f"📊 Model loaded: Finance2_embedding_small_en-V1.5 (FINANCE-SPECIFIC)")
    print(f"🏦 This model is trained on financial texts for better domain separation")
    print(f"💡 Next step: Add sigmoid transformation to boost relevance scores")
    logger.info("✅ FinBERT embeddings initialized")
    print("✅ FinBERT model loaded successfully")
except Exception as e:
    logger.error(f"❌ Error initializing FinBERT embeddings: {str(e)}")
    print(f"❌ Error: Could not load FinBERT. Please install: pip install sentence-transformers")
    raise

print(f"🔄 Embedding {len(docs)} document chunks... (this may take a while)")
logger.info(f"Starting embedding process for {len(docs)} chunks")

# ================================================================
# 🔬 DEBUGGING: Check actual chunk content
# ================================================================
# print("\n" + "="*80)
# print("🔬 DEBUGGING CHUNK CONTENT")
# print("="*80)
# print(f"Total chunks: {len(docs)}")

# if len(docs) >= 3:
#     print(f"\n--- Chunk 0 (First chunk from {docs[0].metadata.get('source', 'Unknown')}) ---")
#     print(f"Length: {len(docs[0].page_content)} characters")
#     print(f"Content preview:\n{docs[0].page_content[:500]}")
    
#     print(f"\n--- Chunk {len(docs)//2} (Middle chunk) ---")
#     mid_idx = len(docs)//2
#     print(f"Source: {docs[mid_idx].metadata.get('source', 'Unknown')}")
#     print(f"Length: {len(docs[mid_idx].page_content)} characters")
#     print(f"Content preview:\n{docs[mid_idx].page_content[:500]}")
    
#     print(f"\n--- Chunk {len(docs)-1} (Last chunk) ---")
#     print(f"Source: {docs[-1].metadata.get('source', 'Unknown')}")
#     print(f"Length: {len(docs[-1].page_content)} characters")
#     print(f"Content preview:\n{docs[-1].page_content[:500]}")
# else:
#     print("⚠️ Not enough chunks to sample")

# # Check for empty or garbage content
# empty_chunks = sum(1 for doc in docs if len(doc.page_content.strip()) < 50)
# print(f"\n⚠️ Warning: {empty_chunks} chunks have less than 50 characters")

# print("="*80)
# print("🔬 END DEBUGGING")
# print("="*80 + "\n")
# ================================================================

# Create DocArrayInMemorySearch vector store with error handling and retry
@retry_on_api_error(max_retries=2, delay=3)
def create_vectorstore():
    """Create vector store with retry on API errors."""
    start_time = time.time()
    
    # from_documents() creates a searchable index in memory from documents and embeddings
    vectorstore = DocArrayInMemorySearch.from_documents(
        documents=docs,
        embedding=embeddings,
    )
    
    elapsed_time = time.time() - start_time
    logger.info(f"✅ Vector store created in {elapsed_time:.1f} seconds")
    
    return vectorstore

try:
    vectorstore = create_vectorstore()
    print(f"✅ Vector store created successfully (DocArrayInMemorySearch)")
    print(f"📊 Total indexed chunks: {len(docs)}")
    logger.info(f"Vector store ready with {len(docs)} indexed chunks")
except Exception as e:
    logger.error(f"❌ Failed to create vector store: {str(e)}")
    print(f"❌ Error: Could not create vector store. Please check your API key and network connection.")
    raise

## 🎯 Conditional RAG System with Smart Fallback

This is the core feature of our Trading Advisor. The system intelligently decides when to use RAG vs LLM using a **Smart Fallback** strategy.

**How it works:**
1. **Check Relevance**: Calculate similarity score between query and documents
2. **Smart Fallback Decision Logic**:
   - If score ≥ 0.65 (RELEVANCE_THRESHOLD) → Use RAG only (high confidence)
   - If 0.50 ≤ score < 0.65 (FALLBACK zone) → Try BOTH RAG and LLM, score both answers, use better one
   - If score < 0.50 → Use LLM only (documents not relevant)
   - If LLM confidence < threshold → Return "Information not available"

**Three output types (what users see):**
- 🟢 **RAG Answer**: Documents are relevant and helpful (may come from Smart Fallback comparison)
- 🔵 **LLM Answer**: LLM's pre-trained knowledge is used (may come from Smart Fallback comparison)
- 🔴 **No Answer**: Neither source has confident information

**Internal Decision Modes:**
- **RAG Mode** (score ≥ 0.65): Directly use RAG
- **Smart Fallback** (0.50 ≤ score < 0.65): Compare both, pick winner (shows as 🟢 or 🔵)
- **LLM Mode** (score < 0.50): Directly use LLM

---
### 📊 Understanding the Three Different Scores

**1. Relevance Score (0.0 - 1.0)**
- **What it measures**: How similar is the query to the documents in the database?
- **How it's calculated**: Cosine similarity between query embedding and document embeddings
- **Purpose**: Decides whether documents are relevant enough to use

**2. RAG Score (1-10)**
- **What it measures**: Quality of the answer generated using documents + LLM
- **How it's calculated**: LLM evaluates the RAG answer on 3 metrics (then averaged):
  - Specificity: How detailed is the answer?
  - Relevance: Does it answer the question?
  - Factuality: Does it contain verifiable facts?

**3. LLM Score (1-10)**
- **What it measures**: Quality of the answer using LLM's pre-trained knowledge only (no documents)
- **How it's calculated**: Same 3 metrics as RAG Score

**Key Insight:**
- **Relevance Score** = "Do documents match the question?" (document quality)
- **LLM Score** = "Is the LLM's answer good?" (answer quality)

In [ ]:
# Cell 13
import json
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from IPython.display import HTML, display
import numpy as np

class ConditionalRAGAdvisor:
    """
    Conditional RAG system that decides when to use documents vs LLM knowledge.
    
    Decision logic:
    - High relevance score (≥ threshold) → Use RAG (documents)
    - Low relevance score (< threshold) → Use LLM only
    - No confident answer → Return "Information not available"
    """
    
    def __init__(self, vectorstore, llm_model, relevance_threshold, fallback_threshold, off_topic_threshold,
                 top_k, llm_confidence_threshold, temperature, sigmoid_midpoint, sigmoid_steepness):
        """
        Initialize the Conditional RAG Advisor with Smart Fallback.
        
        Args:
            vectorstore: Vector database containing documents
            llm_model: Name of LLM model to use (e.g., "gpt-4o-mini")
            relevance_threshold: High confidence threshold for RAG-only (0.0-1.0)
            fallback_threshold: Low threshold for smart fallback (try both RAG and LLM)
            top_k: Number of documents to retrieve
            llm_confidence_threshold: Minimum LLM confidence (0-10) for valid answer
            temperature: LLM temperature (lower = more focused)
        """
        self.vectorstore = vectorstore
        self.relevance_threshold = relevance_threshold
        self.fallback_threshold = fallback_threshold
        self.off_topic_threshold = off_topic_threshold
        self.llm_confidence_threshold = llm_confidence_threshold
        self.top_k = top_k
        self.sigmoid_midpoint = sigmoid_midpoint
        self.sigmoid_steepness = sigmoid_steepness
        
        # Initialize LLM
        self.llm = ChatOpenAI(model=llm_model, temperature=temperature)
        
        # Create custom prompt for RAG chain
        # This forces the LLM to answer based on documents, not refuse due to date restrictions
        rag_prompt_template = """You are a financial analyst assistant. Answer the question based on the provided context documents.

INSTRUCTIONS:
1. Use the information from the context below to answer the question
2. Synthesize information from the context - you don't need exact quotes or lists
3. If the context discusses related topics, provide a helpful answer based on what's available
4. Only say "I don't have enough information" if the context is truly unrelated to the question
5. DO NOT refuse to answer based on date/time restrictions - the documents may contain information about any time period
6. Focus on being helpful - provide the best answer you can from the available context

Context: {context}

Question: {question}

Answer:"""
        
        RAG_PROMPT = PromptTemplate(
            template=rag_prompt_template,
            input_variables=["context", "question"]
        )
        
        # Create RAG chain with custom prompt
        self.rag_chain = RetrievalQA.from_chain_type(
            llm=self.llm,
            chain_type="stuff",
            retriever=vectorstore.as_retriever(search_kwargs={"k": top_k}),
            chain_type_kwargs={"prompt": RAG_PROMPT}
        )
        
    def get_relevance_score(self, query):
        """
        Calculate the RELEVANCE SCORE for a query (0.0 - 1.0).
        
        This score measures: "How similar is the query to the documents in the database?"
        
        Process:
        1. Convert query to embedding vector (using FinBERT)
        2. Compare with all document embeddings using cosine similarity
        3. Calculate average similarity across top-k documents
        
        High score (>0.65) = Documents match query well → Use RAG
        Low score (<0.50) = Documents don't match query → Use LLM knowledge
        
        NOTE: FinBERT embeddings are finance-specific, so they naturally
        provide better separation between finance and non-finance queries
        compared to general-purpose embeddings (like OpenAI).
        """
        # Retrieve documents with similarity scores
        # This compares the query embedding with all document embeddings
        docs_with_scores = self.vectorstore.similarity_search_with_score(query, k=self.top_k)
        
        if not docs_with_scores:
            return 0.0
        
        # DocArrayInMemorySearch returns COSINE SIMILARITY (0-1)
        # Higher score = more similar
        scores = [score for _, score in docs_with_scores]
        
        # Use average similarity for stable, balanced relevance score
        avg_score = sum(scores) / len(scores)
        
        # Apply sigmoid transformation to amplify separation
        # This boosts finance queries (> midpoint) to 0.80-0.99
        # and pushes non-finance queries (< midpoint) to 0.01-0.20
        import math
        sigmoid_score = 1 / (1 + math.exp(-self.sigmoid_steepness * (avg_score - self.sigmoid_midpoint)))
        
        return sigmoid_score
    
    def llm_with_domain_check(self, query):
        """
        Call LLM with domain-aware prompt to handle borderline relevance queries.
        
        This is used when relevance score is low but not zero (0.20-0.50).
        The LLM is instructed to ONLY answer stock trading questions and
        reject off-topic queries politely.
        
        Args:
            query: User's question
            
        Returns:
            LLM response (either an answer or rejection message)
        """
        domain_prompt = f"""You are a STOCK TRADING ADVISOR specializing in financial markets and trading.

IMPORTANT RULES:
1. You can ONLY answer questions about:
   - Stock markets, equities, trading strategies
   - Economic policies, Federal Reserve decisions, interest rates
   - Company earnings, financial analysis, valuations
   - Market trends, predictions, technical/fundamental analysis
   - Investment advice, portfolio management

2. If the question is about OTHER TOPICS (medicine, physics, sports, cooking, history, etc.):
   → Respond EXACTLY: "I'm a stock trading advisor and can only answer questions about financial markets and trading. Your question about [topic] is outside my expertise. Please ask about stock market or trading-related topics."

3. If you're unsure whether it's stock-related:
   → If it has ANY connection to finance/markets → answer it
   → If it's clearly unrelated → reject it politely

Question: {query}

Answer:"""
        
        try:
            response = self.llm.invoke(domain_prompt)
            answer = response.content if hasattr(response, 'content') else str(response)
            logger.info(f"🔍 Domain check LLM response: {answer[:100]}...")
            return answer
        except Exception as e:
            logger.error(f"❌ Error in domain check LLM: {str(e)}")
            return "I apologize, but I encountered an error. Please try again."
    
    def assess_llm_confidence(self, question, answer):
        """
        Ask the LLM to rate its own confidence in the answer.
        
        Uses LLM self-assessment to detect when the model is uncertain or hallucinating.
        
        Args:
            question: The original question
            answer: The LLM-generated answer
            
        Returns:
            Confidence score (0-10), or 0 if assessment fails
        """
        confidence_prompt = f"""Rate your confidence in the following answer on a scale of 0-10:

Question: {question}

Your Answer: {answer}

Instructions:
- If you provided specific, factual information you're confident about: 7-10
- If you provided general knowledge but aren't fully certain: 4-6  
- If you don't have enough information or are guessing: 0-3
- Be honest about your limitations

Respond with ONLY a single number from 0 to 10, nothing else.

Confidence score:"""
        
        try:
            response = self.llm.invoke(confidence_prompt).content.strip()
            # Extract first number found in response
            import re
            match = re.search(r'\d+', response)
            if match:
                confidence = int(match.group())
                # Clamp to 0-10 range
                confidence = max(0, min(10, confidence))
                return confidence
            else:
                return 0
        except Exception as e:
            print(f"Warning: Confidence assessment failed ({e}), assuming low confidence")
            return 0

    def score_answer(self, question, answer):
        """
        Calculate RAG SCORE or LLM SCORE (1-10) for an answer.
        
        This score measures: "How GOOD is this answer?" (NOT "Are documents relevant?")
        
        Evaluates answer quality on 3 dimensions:
        - Specificity: How detailed and specific?
        - Relevance: Does it directly answer the question?
        - Factuality: Does it contain verifiable facts and data?
        
        Final score = average of the 3 metrics
        
        IMPORTANT: This is completely independent of the Relevance Score!
        - Relevance Score (0-1) = "Do documents match query?" 
        - RAG/LLM Score (1-10) = "Is the answer high quality?"
        
        Returns:
            dict with specificity, relevance, and factuality scores (1-10)
        """
        evaluation_prompt = f"""You are an expert evaluator. Score the following answer on these three criteria (scale 1-10):

1. SPECIFICITY: How specific and detailed is the answer? (1=vague, 10=very specific with details)
2. RELEVANCE: How relevant is the answer to the question? (1=off-topic, 10=directly answers question)
3. FACTUALITY: Does the answer contain verifiable facts and data? (1=no facts/opinions only, 10=rich with facts and data)

Question: {question}

Answer: {answer}

Respond ONLY with a JSON object in this exact format (no other text):
{{"specificity": <score>, "relevance": <score>, "factuality": <score>}}"""
        
        try:
            response = self.llm.invoke(evaluation_prompt).content
            # Extract JSON from response (in case there's extra text)
            start_idx = response.find('{')
            end_idx = response.rfind('}') + 1
            json_str = response[start_idx:end_idx]
            scores = json.loads(json_str)
            return scores
        except:
            # Fallback if parsing fails
            return {"specificity": 5, "relevance": 5, "factuality": 5}

    def query(self, question):
        """
        Answer a question using Smart Fallback RAG logic.
        
        Returns a dictionary with the answer and extensive metadata.
        """
        # Step 1: Calculate relevance score
        relevance_score = self.get_relevance_score(question)
        
        # Initialize result dictionary
        result = {
            "answer": None, "mode": None, "relevance_score": relevance_score,
            "llm_confidence": None, "retrieved_docs": [], "fallback_source": None,
            "rag_scores": None, "llm_scores": None
        }
        
        # Step 2: Decide which mode to use based on Smart Fallback logic
        if relevance_score >= self.relevance_threshold:
            # High confidence: Use RAG only
            result["mode"] = "RAG"
            result["answer"] = self.rag_chain.invoke({"query": question})["result"]
            result["retrieved_docs"] = self.vectorstore.similarity_search(question, k=self.top_k)
            
        elif relevance_score >= self.fallback_threshold:
            # Uncertain zone: Try BOTH, score both, and pick the winner
            result["mode"] = "FALLBACK"
            
            # Get RAG answer
            rag_answer = self.rag_chain.invoke({"query": question})["result"]
            result["retrieved_docs"] = self.vectorstore.similarity_search(question, k=self.top_k)
            
            # Get LLM answer
            llm_prompt = f"Answer the following question about stock market and trading:\n\nQuestion: {question}\n\nAnswer:"
            llm_answer = self.llm.invoke(llm_prompt).content
            
            # Assess LLM's confidence
            result["llm_confidence"] = self.assess_llm_confidence(question, llm_answer)
            
            # Decision Logic: If LLM is uncertain, use RAG. Otherwise, score both and compare.
            if result["llm_confidence"] < self.llm_confidence_threshold:
                result["answer"] = rag_answer
                result["fallback_source"] = "RAG"
            else:
                # Score both answers on key metrics
                rag_scores = self.score_answer(question, rag_answer)
                llm_scores = self.score_answer(question, llm_answer)
                
                result["rag_scores"] = rag_scores
                result["llm_scores"] = llm_scores
                
                # Calculate overall scores
                rag_overall = np.mean(list(rag_scores.values()))
                llm_overall = np.mean(list(llm_scores.values()))
                
                # The winner is the one with the higher overall score
                if rag_overall >= llm_overall:
                    result["answer"] = rag_answer
                    result["fallback_source"] = "RAG"
                else:
                    result["answer"] = llm_answer
                    result["fallback_source"] = "LLM"
        else:
            # Low relevance: Need to determine if question is off-topic or stock-related
            # Use 4-tier system with OFF_TOPIC_THRESHOLD
            
            if relevance_score < self.off_topic_threshold:
                # Tier 1: Very low relevance (< 0.20) → Clearly off-topic
                # Immediate rejection without LLM call (saves cost & latency)
                result["mode"] = "OFF_TOPIC"
                result["answer"] = (
                    "I'm a stock trading advisor and can only answer questions about financial markets and trading. "
                    "Your question appears to be outside my area of expertise. "
                    "Please ask about stock market, trading, economic policies, or financial analysis."
                )
                logger.info(f"🚫 Off-topic rejection: relevance={relevance_score:.3f} < {self.off_topic_threshold}")
                
            else:
                # Tier 2: Low but not zero relevance (0.20-0.50) → Borderline
                # Use LLM with domain-aware prompt to decide
                result["mode"] = "LLM_DOMAIN_CHECK"
                result["answer"] = self.llm_with_domain_check(question)
                
                # Assess LLM's confidence in its answer
                result["llm_confidence"] = self.assess_llm_confidence(question, result["answer"])
                
                # If LLM confidence is too low, switch to NO_ANSWER mode
                if result["llm_confidence"] < self.llm_confidence_threshold:
                    result["mode"] = "NO_ANSWER"
                    result["answer"] = f"Information not available. This question cannot be answered with confidence. (LLM confidence: {result['llm_confidence']}/10)"
                
                logger.info(f"🔍 Domain check mode: relevance={relevance_score:.3f}, confidence={result['llm_confidence']}")
        
        return result
    
# Initialize the Conditional RAG Advisor with Smart Fallback
advisor = ConditionalRAGAdvisor(
    vectorstore=vectorstore,
    llm_model=LLM_MODEL,
    relevance_threshold=RELEVANCE_THRESHOLD,
    fallback_threshold=FALLBACK_THRESHOLD,
    off_topic_threshold=OFF_TOPIC_THRESHOLD,
    top_k=TOP_K_DOCUMENTS,
    llm_confidence_threshold=LLM_CONFIDENCE_THRESHOLD,
    temperature=LLM_TEMPERATURE,
    sigmoid_midpoint=SIGMOID_MIDPOINT,
    sigmoid_steepness=SIGMOID_STEEPNESS
)

print("✅ Conditional RAG Advisor (Smart Fallback) initialized successfully!")
print(f"📊 Configuration: Model={LLM_MODEL}, RAG Threshold={RELEVANCE_THRESHOLD}, Fallback={FALLBACK_THRESHOLD}, Confidence={LLM_CONFIDENCE_THRESHOLD}, Top-K={TOP_K_DOCUMENTS}")

## 📊 Enhanced Query Function with Monitoring

Now we'll wrap the advisor's query method with monitoring to track:
- ⏱️ Response time
- 💰 Token usage and cost
- 📊 Mode distribution
- ⚠️ Errors and retries

This wrapper will be used for all subsequent queries to automatically log performance metrics.

In [ ]:
# Cell 14

def monitored_query(advisor, question, track_cost=True):
    """
    Enhanced query function with monitoring and error handling.
    
    Args:
        advisor: ConditionalRAGAdvisor instance
        question: Question to ask
        track_cost: Whether to track token usage and cost
    
    Returns:
        Result dictionary with answer and metadata
    """
    start_time = time.time()
    result = None
    error_occurred = False
    
    try:
        logger.info(f"🔍 Processing query: {question[:50]}...")
        
        # Call the advisor's query method with retry on API errors
        @retry_on_api_error(max_retries=2, delay=2)
        def query_with_retry():
            return advisor.query(question)
        
        result = query_with_retry()
        
        # Calculate response time
        response_time = time.time() - start_time
        result['response_time'] = response_time
        
        # Track token usage if enabled
        if track_cost:
            # Estimate token usage based on input/output length
            input_text = question
            output_text = result['answer']
            input_tokens, output_tokens = cost_tracker.add_tokens(input_text, output_text)
            
            result['input_tokens'] = input_tokens
            result['output_tokens'] = output_tokens
        
        # Log the result
        mode = result['mode']
        logger.info(f"✅ Query completed in {response_time:.2f}s | Mode: {mode} | Relevance: {result['relevance_score']:.3f}")
        
        # Track mode for experiment logging
        if hasattr(experiment_tracker, 'current_experiment') and experiment_tracker.current_experiment:
            experiment_tracker.log_query(response_time, mode)
        
        return result
        
    except Exception as e:
        error_occurred = True
        response_time = time.time() - start_time
        
        logger.error(f"❌ Query failed after {response_time:.2f}s: {type(e).__name__} - {str(e)}")
        
        # Return error result
        return {
            'mode': 'ERROR',
            'answer': f"Sorry, an error occurred while processing your question: {str(e)}",
            'relevance_score': 0.0,
            'response_time': response_time,
            'error': str(e)
        }

# Create a convenience function that uses the global advisor
def ask(question):
    """
    Convenient function to ask a question with monitoring.
    
    Usage:
        result = ask("What happened in the market?")
    """
    return monitored_query(advisor, question)

print("✅ Monitored query function ready!")
print("💡 Use ask(question) for quick queries with automatic monitoring")

## 🧪 Testing the Conditional RAG System

Let's test the advisor with sample questions to see how it decides between RAG and LLM modes.

---
### 📊 Quick Reference: The Three Scores

```
┌─────────────────────────────────────────────────────────────────┐
│  RELEVANCE SCORE (0.0 - 1.0)                                    │
│  Question: "Do documents match the query?"                      │
│  • 0.85 → Documents highly relevant to query ✓                  │
│  • 0.54 → Documents somewhat match (uncertain zone)             │
│  • 0.30 → Documents not relevant to query ✗                     │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│  RAG SCORE (1-10)                                               │
│  Question: "How good is the RAG answer?"                        │
│  • Measures: Specificity + Relevance + Factuality               │
│  • 8.5 → High quality answer with specific facts from docs      │
│  • 6.0 → Decent answer but lacks detail                         │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│  LLM SCORE (1-10)                                               │
│  Question: "How good is the LLM-only answer?"                   │
│  • Measures: Same as RAG Score                                  │
│  • 8.0 → Good answer using LLM's pre-trained knowledge          │
│  • 4.0 → LLM doesn't have good info on this topic               │
└─────────────────────────────────────────────────────────────────┘

💡 KEY INSIGHTS:
   1. Low Relevance + High LLM Score = Documents irrelevant BUT LLM knows the answer
   2. High Relevance + Low RAG Score = Documents match BUT answer quality is poor
```

In [ ]:
# Cell 15
# HTML display function for better formatting
def show_response(result, question):
    """
    Display the advisor's response in a formatted HTML box.
    Shows score comparison instead of LLM confidence for better transparency.
    """
    mode = result['mode']
    answer = result['answer']
    relevance_score = result['relevance_score']
    fallback_source = result.get('fallback_source')
    rag_scores = result.get('rag_scores')
    llm_scores = result.get('llm_scores')
    
    # Color coding based on mode
    if mode == "RAG":
        color = "#4caf50"  # Green
        icon = "🟢"
        mode_text = "RAG Mode (Using Documents)"
    elif mode == "FALLBACK":
        # In Fallback mode, color depends on the final answer source
        if fallback_source == "RAG":
            color = "#4caf50"  # Green
            icon = "🟢"
            mode_text = "RAG Answer (from Smart Fallback)"
        else: # LLM was chosen
            color = "#2196f3"  # Blue
            icon = "🔵"
            mode_text = "LLM Answer (from Smart Fallback)"
    elif mode == "LLM_DOMAIN_CHECK":
        # Check if LLM rejected the query as off-topic (domain rejection)
        # Be specific: only flag as off-topic if BOTH conditions met
        domain_rejection_keywords = [
            "stock trading advisor",
            "outside my expertise", 
            "outside my area"
        ]
        # Strong indicator of domain rejection
        has_domain_rejection = any(keyword.lower() in answer.lower() for keyword in domain_rejection_keywords)
        # Also mentions asking about stock/trading topics (redirecting user)
        has_redirection = "ask about stock" in answer.lower() or "trading-related topics" in answer.lower()
        
        is_off_topic_rejection = has_domain_rejection and has_redirection
        
        if is_off_topic_rejection:
            color = "#f44336"  # Red (rejected - not stock related)
            icon = "🔴"
            mode_text = "Off-Topic Question (Rejected by Domain Check)"
        else:
            color = "#ff9800"  # Orange (answered - stock related, even if with caveats)
            icon = "🟠"
            mode_text = "LLM Answer (Stock-Related, Low Document Relevance)"
    elif mode == "OFF_TOPIC":
        color = "#f44336"  # Red (auto-rejected - clearly off-topic)
        icon = "🔴"
        mode_text = "Off-Topic (Auto-Rejected)"
    elif mode == "NO_ANSWER":
        color = "#f44336"  # Red
        icon = "🔴"
        mode_text = "No Answer Available (Low Confidence)"
    elif mode == "ERROR":
        color = "#f44336"  # Red
        icon = "❌"
        mode_text = "System Error"
    else:
        color = "#9e9e9e"  # Grey
        icon = "⚪"
        mode_text = f"Unknown Mode: {mode}"
    
    # Build metadata line with relevance score
    metadata = f"{icon} <strong>{mode_text}</strong> | Relevance Score: {relevance_score:.3f}"
    
    # For FALLBACK mode, show score comparison instead of LLM confidence
    if mode == "FALLBACK" and rag_scores and llm_scores:
        # Calculate overall scores (average of specificity, relevance, factuality)
        rag_overall = sum(rag_scores.values()) / len(rag_scores)
        llm_overall = sum(llm_scores.values()) / len(llm_scores)
        
        # Show comparison
        if fallback_source == "RAG":
            metadata += f" | <span style='color:#4caf50;'><strong>RAG Score: {rag_overall:.1f}</strong></span> > LLM Score: {llm_overall:.1f}"
        else:
            metadata += f" | RAG Score: {rag_overall:.1f} < <span style='color:#2196f3;'><strong>LLM Score: {llm_overall:.1f}</strong></span>"
    
    html = f"""
    <div style="
        background-color:#f9f9f9;
        border-left: 6px solid {color};
        padding: 15px;
        margin: 15px 0;
        border-radius: 4px;
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Arial, sans-serif;">
        <div style="color:#333; font-size:14px; margin-bottom:10px;">
            <strong>Question:</strong> {question}
        </div>
        <div style="color:#666; font-size:13px; margin-bottom:10px;">
            {metadata}
        </div>
        <div style="color:#111; font-size:14px; line-height:1.6; white-space: pre-wrap;">
            <strong>Answer:</strong><br>{answer}
        </div>
    </div>
    """
    display(HTML(html))

# Test with the configured test questions
print("🧪 Testing Conditional RAG Advisor with sample questions...\n")

# Start experiment tracking
experiment_tracker.start_experiment({
    'relevance_threshold': RELEVANCE_THRESHOLD,
    'fallback_threshold': FALLBACK_THRESHOLD,
    'off_topic_threshold': OFF_TOPIC_THRESHOLD,
    'llm_confidence_threshold': LLM_CONFIDENCE_THRESHOLD,
    'sigmoid_midpoint': SIGMOID_MIDPOINT,
    'sigmoid_steepness': SIGMOID_STEEPNESS,
    'chunk_size': CHUNK_SIZE,
    'top_k': TOP_K_DOCUMENTS,
    'temperature': LLM_TEMPERATURE
})

# Reset cost tracker for this experiment
cost_tracker.reset()

# Store relevance scores for averaging later
relevance_scores = []

for i, question in enumerate(TEST_QUERIES, 1):
    print(f"\n{'='*80}")
    print(f"Test {i}/{len(TEST_QUERIES)}")
    print(f"{'='*80}")
    
    # Use monitored_query instead of advisor.query()
    result = monitored_query(advisor, question)
    show_response(result, question)
    
    # Store relevance score
    if 'relevance_score' in result:
        relevance_scores.append(result['relevance_score'])
    
    # Display performance metrics
    if 'response_time' in result:
        print(f"⏱️ Response time: {result['response_time']:.2f}s")
    if 'input_tokens' in result and 'output_tokens' in result:
        print(f"🎫 Tokens: {result['input_tokens']} input + {result['output_tokens']} output = {result['input_tokens'] + result['output_tokens']} total")

# Display experiment summary
print(f"\n{'='*80}")
print("📊 EXPERIMENT SUMMARY")
print(f"{'='*80}")
cost_summary = cost_tracker.get_summary()
print(f"💰 Total tokens: {cost_summary['total_tokens']:,}")
print(f"💵 Estimated cost: ${cost_summary['estimated_cost_usd']:.4f}")
print(f"📊 Mode distribution:")
for mode, count in experiment_tracker.current_experiment['mode_counts'].items():
    if count > 0:
        print(f"   - {mode}: {count}")

## 📊 Quantitative Evaluation: RAG vs LLM Comparison

Now let's compare answers **with RAG** vs **without RAG** to evaluate the system's performance.

**Evaluation Metrics:**
1. **Specificity Score**: How specific/detailed is the answer? (1-10)
2. **Relevance Score**: How relevant is the answer to the question? (1-10)
3. **Factuality Score**: Does the answer contain verifiable facts? (1-10)
4. **Answer Length**: Number of words in the answer

We'll use the LLM itself to score the answers for consistency.


In [ ]:
# Cell 17

import json
import pandas as pd

class RAGEvaluator:
    """
    Evaluates and compares RAG vs LLM-only responses.
    Uses LLM-as-a-judge for consistent scoring.
    """
    
    def __init__(self, llm, rag_chain):
        """
        Args:
            llm: ChatOpenAI instance for evaluation
            rag_chain: The advisor's RAG chain (with custom prompt)
        """
        self.llm = llm
        self.rag_chain = rag_chain
        
    def get_llm_only_answer(self, question):
        """
        Get answer using LLM only (no RAG).
        """
        prompt = f"""Answer the following question about stock market and trading using only your pre-trained knowledge:

Question: {question}

Answer:"""
        return self.llm.invoke(prompt).content
    
    def get_rag_answer(self, question):
        """
        Get answer using RAG (documents + LLM).
        Uses the advisor's RAG chain to ensure consistent behavior.
        """
        return self.rag_chain.invoke({"query": question})["result"]
    
    def score_answer(self, question, answer):
        """
        Score an answer on multiple metrics using LLM-as-a-judge.
        
        Returns:
            dict with specificity, relevance, and factuality scores (1-10)
        """
        evaluation_prompt = f"""You are an expert evaluator. Score the following answer on these three criteria (scale 1-10):

1. SPECIFICITY: How specific and detailed is the answer? (1=vague, 10=very specific with details)
2. RELEVANCE: How relevant is the answer to the question? (1=off-topic, 10=directly answers question)
3. FACTUALITY: Does the answer contain verifiable facts and data? (1=no facts/opinions only, 10=rich with facts and data)

Question: {question}

Answer: {answer}

Respond ONLY with a JSON object in this exact format (no other text):
{{"specificity": <score>, "relevance": <score>, "factuality": <score>}}"""
        
        try:
            response = self.llm.invoke(evaluation_prompt).content
            # Extract JSON from response (in case there's extra text)
            start_idx = response.find('{')
            end_idx = response.rfind('}') + 1
            json_str = response[start_idx:end_idx]
            scores = json.loads(json_str)
            return scores
        except:
            # Fallback if parsing fails
            return {"specificity": 5, "relevance": 5, "factuality": 5}
    
    def evaluate_comparison(self, question):
        """
        Compare RAG vs LLM-only for a single question.
        
        Returns:
            dict with both answers and their scores
        """
        # Get both answers
        llm_answer = self.get_llm_only_answer(question)
        rag_answer = self.get_rag_answer(question)
        
        # Score both answers
        llm_scores = self.score_answer(question, llm_answer)
        rag_scores = self.score_answer(question, rag_answer)
        
        # Calculate additional metrics
        llm_word_count = len(llm_answer.split())
        rag_word_count = len(rag_answer.split())
        
        return {
            "question": question,
            "llm_answer": llm_answer,
            "rag_answer": rag_answer,
            "llm_specificity": llm_scores["specificity"],
            "rag_specificity": rag_scores["specificity"],
            "llm_relevance": llm_scores["relevance"],
            "rag_relevance": rag_scores["relevance"],
            "llm_factuality": llm_scores["factuality"],
            "rag_factuality": rag_scores["factuality"],
            "llm_word_count": llm_word_count,
            "rag_word_count": rag_word_count,
        }

# Initialize evaluator with advisor's RAG chain
evaluator = RAGEvaluator(advisor.llm, advisor.rag_chain)

# Run evaluation on all test questions
print("📊 Running quantitative evaluation (this will take a few minutes)...\n")
evaluation_results = []

for i, question in enumerate(TEST_QUERIES, 1):
    print(f"Evaluating {i}/{len(TEST_QUERIES)}: {question[:50]}...")
    result = evaluator.evaluate_comparison(question)
    evaluation_results.append(result)
    print("  ✓ Complete")

print("\n✅ Evaluation complete!")

## 📈 Evaluation Results Visualization
Let's visualize the comparison between RAG and LLM-only responses.

In [ ]:
# Cell 19

# Convert results to DataFrame for easy analysis
df = pd.DataFrame(evaluation_results)

# Calculate average scores
avg_llm_specificity = df['llm_specificity'].mean()
avg_rag_specificity = df['rag_specificity'].mean()
avg_llm_relevance = df['llm_relevance'].mean()
avg_rag_relevance = df['rag_relevance'].mean()
avg_llm_factuality = df['llm_factuality'].mean()
avg_rag_factuality = df['rag_factuality'].mean()
avg_llm_words = df['llm_word_count'].mean()
avg_rag_words = df['rag_word_count'].mean()

# Calculate overall score (average of all three metrics)
df['llm_overall'] = (df['llm_specificity'] + df['llm_relevance'] + df['llm_factuality']) / 3
df['rag_overall'] = (df['rag_specificity'] + df['rag_relevance'] + df['rag_factuality']) / 3

avg_llm_overall = df['llm_overall'].mean()
avg_rag_overall = df['rag_overall'].mean()

# Display summary statistics
print("=" * 80)
print("📊 EVALUATION SUMMARY")
print("=" * 80)
print()
print(f"{'Metric':<25} {'LLM Only':<15} {'RAG':<15} {'Difference':<15}")
print("-" * 80)
print(f"{'Specificity (1-10)':<25} {avg_llm_specificity:<15.2f} {avg_rag_specificity:<15.2f} {avg_rag_specificity - avg_llm_specificity:+.2f}")
print(f"{'Relevance (1-10)':<25} {avg_llm_relevance:<15.2f} {avg_rag_relevance:<15.2f} {avg_rag_relevance - avg_llm_relevance:+.2f}")
print(f"{'Factuality (1-10)':<25} {avg_llm_factuality:<15.2f} {avg_rag_factuality:<15.2f} {avg_rag_factuality - avg_llm_factuality:+.2f}")
print(f"{'Overall Score (1-10)':<25} {avg_llm_overall:<15.2f} {avg_rag_overall:<15.2f} {avg_rag_overall - avg_llm_overall:+.2f}")
print(f"{'Average Word Count':<25} {avg_llm_words:<15.1f} {avg_rag_words:<15.1f} {avg_rag_words - avg_llm_words:+.1f}")
print()

# Calculate improvement percentage (OLD METHOD - for reference)
# This calculates: (avg of RAG scores - avg of LLM scores) / avg of LLM scores
improvement_aggregate = ((avg_rag_overall - avg_llm_overall) / avg_llm_overall) * 100

# We'll use the per-query method (calculated below) as the primary metric
# This is more accurate as it calculates improvement for each query individually

# Calculate improvement statistics per question (for stability analysis)
# THIS IS THE PRIMARY METHOD - more accurate and consistent
import numpy as np

# Calculate improvement percentage for each question
individual_improvements = []
for idx, row in df.iterrows():
    if row['llm_overall'] > 0:  # Avoid division by zero
        improvement_pct = ((row['rag_overall'] - row['llm_overall']) / row['llm_overall']) * 100
        individual_improvements.append(improvement_pct)

# Calculate statistics
if len(individual_improvements) > 0:
    improvement = np.mean(individual_improvements)  # PRIMARY METRIC (average of improvements)
    improvement_std = np.std(individual_improvements)
    improvement_min = min(individual_improvements)
    improvement_max = max(individual_improvements)
else:
    improvement = None
    improvement_std = None
    improvement_min = None
    improvement_max = None

# Display results
print(f"🎯 RAG Overall Improvement: {improvement:+.1f}%" if improvement is not None else "🎯 RAG Overall Improvement: N/A")
print(f"   (Calculated as: average of per-query improvements)")
print(f"   Alternative method (aggregate): {improvement_aggregate:+.1f}%")
print()

# Display detailed results table
print("=" * 80)
print("📋 DETAILED RESULTS BY QUESTION")
print("=" * 80)
print()

# Create a formatted table
result_table = df[['question', 'llm_overall', 'rag_overall']].copy()
result_table['improvement'] = result_table['rag_overall'] - result_table['llm_overall']
result_table['winner'] = result_table['improvement'].apply(lambda x: 'RAG ✓' if x > 0 else ('LLM ✓' if x < 0 else 'Tie'))

display(result_table)

# Display stability metrics (already calculated above)
if improvement is not None:
    print(f"\n{'='*80}")
    print("📊 IMPROVEMENT STABILITY ANALYSIS")
    print(f"{'='*80}")
    print(f"Average Improvement:  {improvement:+.1f}%")
    print(f"Standard Deviation:   {improvement_std:.2f}%")
    print(f"Minimum Improvement:  {improvement_min:+.1f}%")
    print(f"Maximum Improvement:  {improvement_max:+.1f}%")
    print(f"Range (Max - Min):    {improvement_max - improvement_min:.1f}%")
    print()
    
    # Stability interpretation
    if improvement_std < 10:
        stability = "🟢 Very Stable"
    elif improvement_std < 20:
        stability = "🟡 Moderately Stable"
    else:
        stability = "🔴 Unstable"
    
    print(f"Stability Rating: {stability}")
    print(f"💡 Lower std = more consistent across different queries")
    print()
    print(f"📝 Note: Two calculation methods:")
    print(f"   - Per-query method (used): {improvement:+.1f}% (average of individual improvements)")
    print(f"   - Aggregate method: {improvement_aggregate:+.1f}% (improvement of averages)")
    print(f"   - We use per-query method as it's more accurate for stability analysis")

# Save experiment results to CSV
print(f"\n{'='*80}")
print("💾 SAVING EXPERIMENT RESULTS")
print(f"{'='*80}")

# Calculate average relevance score from test queries
avg_relevance_score = sum(relevance_scores) / len(relevance_scores) if relevance_scores else None

experiment_tracker.save_experiment(
    improvement_pct=improvement,
    improvement_std=improvement_std,
    improvement_min=improvement_min,
    improvement_max=improvement_max,
    avg_relevance_score=avg_relevance_score,
    avg_rag_score=avg_rag_overall,
    avg_llm_score=avg_llm_overall,
    avg_rag_specificity=avg_rag_specificity,
    avg_rag_relevance=avg_rag_relevance,
    avg_rag_factuality=avg_rag_factuality,
    avg_llm_specificity=avg_llm_specificity,
    avg_llm_relevance=avg_llm_relevance,
    avg_llm_factuality=avg_llm_factuality,
    notes=f"Test run with {len(TEST_QUERIES)} questions"
)

print(f"\n✅ Experiment results saved to: hyperparameter_experiments.csv")
print(f"📊 Saved metrics:")
print(f"   - Avg Improvement: {improvement:+.1f}% (Std: {improvement_std:.2f}%, Min: {improvement_min:+.1f}%, Max: {improvement_max:+.1f}%)" if improvement_std is not None else f"   - Avg Improvement: {improvement:+.1f}%")
print(f"   - Avg Relevance Score: {avg_relevance_score:.3f}" if avg_relevance_score else "   - Avg Relevance Score: N/A")
print(f"   - Avg RAG Score: {avg_rag_overall:.2f}, Avg LLM Score: {avg_llm_overall:.2f}")
print(f"   - RAG (S/R/F): {avg_rag_specificity:.1f}/{avg_rag_relevance:.1f}/{avg_rag_factuality:.1f}")
print(f"   - LLM (S/R/F): {avg_llm_specificity:.1f}/{avg_llm_relevance:.1f}/{avg_llm_factuality:.1f}")
print(f"\n💡 Tip: Open the CSV file in Excel to compare different hyperparameter settings!")
print(f"📈 Pro Tip: Use 'improvement_std' column to find stable settings (lower std = more consistent)")


## 🔍 Side-by-Side Answer Comparison
Let's see the actual answers from both RAG and LLM-only modes.

In [ ]:
# Cell 21

def show_comparison(result):
    """
    Display side-by-side comparison of RAG vs LLM answers.
    Shows overall score (average of 3 metrics) plus individual scores.
    """
    question = result['question']
    llm_answer = result['llm_answer']
    rag_answer = result['rag_answer']
    
    # Calculate overall scores (average of the 3 metrics)
    llm_overall = (result['llm_specificity'] + result['llm_relevance'] + result['llm_factuality']) / 3
    rag_overall = (result['rag_specificity'] + result['rag_relevance'] + result['rag_factuality']) / 3
    
    # Format: Overall Score = X.X (Spec: X, Rel: X, Fact: X)
    llm_scores = f"Overall Score = {llm_overall:.1f} (Spec: {result['llm_specificity']}, Rel: {result['llm_relevance']}, Fact: {result['llm_factuality']})"
    rag_scores = f"Overall Score = {rag_overall:.1f} (Spec: {result['rag_specificity']}, Rel: {result['rag_relevance']}, Fact: {result['rag_factuality']})"
    
    html = f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Arial, sans-serif; margin: 20px 0;">
        <div style="background-color:#f5f5f5; padding:15px; border-radius:8px; margin-bottom:20px;">
            <h3 style="margin:0; color:#333;">Question:</h3>
            <p style="font-size:16px; color:#000; margin:10px 0 0 0;">{question}</p>
        </div>
        
        <div style="display:flex; gap:20px;">
            <!-- LLM Only -->
            <div style="flex:1; background-color:#fff; border:2px solid #2196f3; border-radius:8px; padding:15px;">
                <h4 style="margin:0 0 10px 0; color:#2196f3;">🔵 LLM Only</h4>
                <p style="font-size:12px; color:#666; margin:0 0 10px 0;">{llm_scores}</p>
                <div style="font-size:14px; line-height:1.6; color:#333; white-space:pre-wrap;">{llm_answer}</div>
            </div>
            
            <!-- RAG -->
            <div style="flex:1; background-color:#fff; border:2px solid #4caf50; border-radius:8px; padding:15px;">
                <h4 style="margin:0 0 10px 0; color:#4caf50;">🟢 RAG (Documents + LLM)</h4>
                <p style="font-size:12px; color:#666; margin:0 0 10px 0;">{rag_scores}</p>
                <div style="font-size:14px; line-height:1.6; color:#333; white-space:pre-wrap;">{rag_answer}</div>
            </div>
        </div>
    </div>
    """
    display(HTML(html))

# Display comparison for each question
for i, result in enumerate(evaluation_results, 1):
    print(f"\n{'='*100}")
    print(f"Question {i}/{len(evaluation_results)}")
    print(f"{'='*100}\n")
    show_comparison(result)

## 💬 Interactive Q&A Interface
Now you can ask your own questions to the Trading Advisor!

In [ ]:
# Cell 23

import ipywidgets as widgets
from IPython.display import clear_output

# Create interactive widgets
question_input = widgets.Textarea(
    value='',
    placeholder='Ask a question about the stock market (e.g., "What happened in the market last week?")',
    description='Your Question:',
    layout=widgets.Layout(width='80%', height='80px'),
    style={'description_width': '120px'}
)

submit_button = widgets.Button(
    description='Get Answer',
    button_style='primary',
    icon='search'
)

output_area = widgets.Output()

def on_submit_click(b):
    """
    Handle submit button click.
    """
    with output_area:
        clear_output()
        
        question = question_input.value.strip()
        
        if not question:
            print("⚠️ Please enter a question!")
            return
        
        print(f"🔍 Processing your question...\n")
        
        # Get answer from advisor with monitoring
        result = monitored_query(advisor, question)
        
        # Display the response
        show_response(result, question)
        
        # Display performance metrics
        if 'response_time' in result:
            print(f"\n⏱️ Response time: {result['response_time']:.2f}s")
        if 'input_tokens' in result and 'output_tokens' in result:
            total_cost = (result['input_tokens'] * 0.150 + result['output_tokens'] * 0.600) / 1_000_000
            print(f"🎫 Tokens: {result['input_tokens'] + result['output_tokens']} total")
            print(f"💵 Cost: ${total_cost:.4f}")
        
        # Show retrieved documents if RAG mode was used
        if result['mode'] == 'RAG' and result['retrieved_docs']:
            print("\n📄 Retrieved Documents:")
            print("-" * 80)
            for i, doc in enumerate(result['retrieved_docs'][:2], 1):  # Show top 2 docs
                preview = doc.page_content[:200].replace('\n', ' ')
                print(f"\n{i}. {preview}...")

# Attach button click handler
submit_button.on_click(on_submit_click)

# Display the interface
print("💬 Interactive Trading Advisor Interface")
print("=" * 80)
print("\nAsk questions about stock market, trading advice, or economic events.\n")

display(question_input)
display(submit_button)
display(output_area)


## 🎓 Summary and Key Learnings

### What We Built
✅ **Conditional RAG System**: Intelligently decides when to use documents vs LLM knowledge  
✅ **Relevance Scoring**: Calculates similarity scores to make smart decisions  
✅ **Quantitative Evaluation**: Compares RAG vs LLM with multiple metrics  
✅ **Interactive Interface**: Easy-to-use Q&A system for trading advice  

---

### Key Components
1. **Document Loading**: Multiple PDFs loaded from `docs/` folder
2. **Vector Store**: DocArrayInMemorySearch for fast similarity search
3. **ConditionalRAGAdvisor**: Core class that implements decision logic
4. **RAGEvaluator**: Evaluation system using LLM-as-a-judge
5. **Interactive UI**: ipywidgets-based interface

---

### How Conditional RAG Works

```
Query → Calculate Relevance Score
         ↓
    Three Decision Paths:
         ↓
    ┌────┴────┐
    │ Score?  │
    └────┬────┘
         │
    ├──────────┼──────────┤
    ↓          ↓          ↓
  ≥0.65    0.50-0.65    <0.50
    ↓          ↓          ↓
RAG Mode   FALLBACK   LLM Mode
🟢 Green   (Compare)   🔵 Blue
               ↓
          Winner gets
            🟢 or 🔵
```

**Key Point:** Users only see 🟢 Green or 🔵 Blue outputs. Smart Fallback is internal!

---

### Configuration Settings You Can Adjust
- **RELEVANCE_THRESHOLD**: Higher = stricter document matching (default: 0.65)
- **TOP_K_DOCUMENTS**: Number of documents to retrieve (default: 4)
- **CHUNK_SIZE**: Size of document chunks (default: 800)
- **LLM_MODEL**: GPT model to use (default: "gpt-4o-mini")
- **LLM_TEMPERATURE**: Response creativity (default: 0.3)

---

### Next Steps
1. **Add More Documents**: Place new PDFs in `docs/` folder and rerun
2. **Adjust Threshold**: Experiment with RELEVANCE_THRESHOLD for different behaviors
3. **Try Different Models**: Test with gpt-4 or other models
4. **Expand Evaluation**: Add more test questions
5. **Deploy**: Turn this into a web app using Streamlit or Gradio

---

### 📚 Learning Points

**For Entry-to-Mid Level AI/ML Engineers:**

- ✅ **RAG Pattern**: You now understand how to implement Retrieval Augmented Generation
- ✅ **Conditional Logic**: Decision-making based on relevance scores
- ✅ **Evaluation**: Using LLM-as-a-judge for consistent scoring
- ✅ **Production Readiness**: Clean class design, configurability, error handling
- ✅ **Best Practices**: Proper docstrings, type hints implied, modular code

This project demonstrates **Entry-to-Mid level proficiency** in:
- RAG systems
- LangChain framework
- Vector search
- System evaluation
- Interactive UI development